# Standby Duty Schedule

not down all sources!

## Current Situation

- always 90 standby drives (baseline model)

In [ ]:
# next steps
# o TODOs below
# o add to word in parallel
# o push to github
# o push pdfs to drive for romania

## Exploratory Data Analysis

| Column | Datatype | Description |
| --- | --- | --- |
| date | string | yyyy-mm-dd |
| n_sick | int | amount of sick drivers |
| calls | float | emergency calls |
| n_duty | int | amount of **on-duty** drivers |
| n_sby | int | amount of available **standby** drivers |
| sby_need | float | amount of activated **standby** drivers |
| dafted | float | drafted **off-duty drivers** if standby drivers are not enough |

In [ ]:
# imports 
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib ipympl

# import dataset
df = pd.read_csv("sickness_table.csv")

# delete indexing variable of dataset
if "Unnamed: 0" in df.columns:
    del df["Unnamed: 0"]

### Evaluation of Data Quality

In [ ]:
# check if there are any empty cells
if not df.isnull().any().any():
    print("There are no empty cells in the dataset.")

# check if all floats are actually whole numbers (x.0) and parse to int
float_cols = ["calls", "sby_need", "dafted"]
parse_floats_to_ints = True
for col in float_cols:
    if not all(df[col].apply(float.is_integer)):
        print(f"Column {col} contains non-integer floats.")
        parse_floats_to_ints = False
if parse_floats_to_ints:
    print("All float columns contain only whole numbers. Parsing to int.")
    df[float_cols] = df[float_cols].astype(int)

# check if all dates follow yyyy-mm-dd schema
if all(df["date"].str.match(r"\d{4}-\d{2}-\d{2}")):
    print("All dates follow the yyyy-mm-dd schema.")
# check if data for all days is available
start_date = df["date"].min()
end_date = df["date"].max()
if not any(pd.date_range(start=start_date, end=end_date).difference(pd.to_datetime(df["date"]))):
    print("Data is available for all days.")

In [ ]:
# create profile report
profile = ProfileReport(df, title="Sickness Table Report")
profile.to_notebook_iframe()

### Feature generation

then: - visualize important connections/correlation (easily understandable, also show quality of data)

In [ ]:
# create additional features: d, m , y and delete original date column
df[["year", "month", "day"]] = df["date"].str.split("-", expand=True).astype(int)

# create additional feature: n_work (actually working drivers)
df["n_work"] = df["n_duty"] - df["n_sick"] + df["sby_need"]

# percentage of sick drivers since n_duty changes
df["perc_sick"] = df["n_sick"] / df["n_duty"]

# TODO:
# get trend, seasonal and noise components of calls
# calls_trend, calls_seas, calls_noise

# get trend, seasonal and noise components of perc_sick
# perc_sick_trend, perc_sick_seas, perc_sick_noise

In [ ]:
# general plotting
df_plot = df.drop(columns=["n_sby", "month", "day", "date"])  # w/o uninformative columns
fig, axs = plt.subplots(len(df_plot.columns) + 1, 1, figsize=(10, 2 * len(df_plot.columns)), sharex=True)
for i, ax in enumerate(axs):
    if i < len(axs) - 1:
        ax.plot(df_plot.iloc[:, i])
        ax.set_title(df_plot.columns[i])
    else:
        # evaluation metric: percentage of sby_need compared to n_sby
        ax.plot(100 * df["sby_need"] / df["n_sby"])
        ax.hlines(100, xmin=df.index.min(), xmax=df.index.max(), colors="r", linestyles="dashed")
        ax.set_title("Percentage of sby_need compared to n_sby")
fig.tight_layout()

In [ ]:
# plot relationship between n_work and calls for each n_duty
# better than plot in sns pairplot since noise got reduced
n_duties = df["n_duty"].unique()
fig, ax = plt.subplots(1, 1, figsize=(6, 4), sharex=True)
for n_duty in n_duties:
    ax.scatter(
        df["n_work"].where(df["n_duty"] == n_duty),
        df["calls"].where(df["n_duty"] == n_duty),
        marker="x",
        label=f"n_duty: {n_duty}",
    )
ax.scatter(
    df["n_work"].where(df["sby_need"] > 0),
    df["calls"].where(df["sby_need"] > 0),
    alpha=0.4,
    marker=".",
    color="red",
    label="sby_need > 0",
)
ax.legend()
ax.set_xlabel("n_work")
ax.set_ylabel("calls")
ax.grid()
fig.tight_layout()

In [ ]:
# calculate correlation between n_work and calls w/o sby_need for each n_duty
print("corr between n_work and calls (only if standby drivers were needed)")
for n_duty in n_duties:
    indices = df.index[(df["sby_need"] > 0) & (df["n_duty"] == n_duty)]
    n_work = df.loc[indices, "n_work"]
    calls = df.loc[indices, "calls"]
    r = np.corrcoef(n_work, calls)
    print(f"\tFor n_duty={n_duty} the corr is {r[1, 0]}")

In [ ]:
# calculate correlation between n_work and n_sick w/o sby_need for each n_duty
print("corr between n_work and n_sick (only if no standby drivers were needed)")
for n_duty in n_duties:
    indices = df.index[(df["sby_need"] <= 0) & (df["n_duty"] == n_duty)]
    n_work = df.loc[indices, "n_work"]
    n_sick = df.loc[indices, "n_sick"]
    r = np.corrcoef(n_work, n_sick)
    print(f"\tFor n_duty={n_duty} the corr is {r[1, 0]}")

In [ ]:
# TODO: more plots
# - idea: only predict calls and go from there? (for that only date and previous calls are necessary (everything else is not related))
#   (prediction needs to be on a daily basis (look that up in pdfs))
#   (then linear regression?)
#   but date information is probably needed!!! (but also include trend, seasonal and noise)
#   relevant columns: calls_trend, calls_seas, calls_resid, year, month, day, perc_sick_trend, perc_sick_seas, perc_sick_resid
# o read Kurs-Skript for all necessary steps (more variables?, how to asses quality?, cleaning?)

# plot correlation matrix
# except for n_sby since it's always 90
df.drop(columns="n_sby").corr().style.background_gradient(cmap="coolwarm", axis=None, vmin=-1, vmax=1)

In [ ]:
# Findings
# general:
#   - for n_work above n_duty-n_sick correlation between calls and n_work is very strong
#     (for lower n_work there is a lot of noise due to n_sick, but upper limit follows correlation)
#   - n_sick and sby_need/dafted are not related
#   - sby_needed is the same as drafted just w/ offset of n_sby
#   - n_duty got elevated every year
# correlations:
#   - the more n_duty, the more n_sick (obvious)
#   - the more n_duty, the more n_work (obvious)
#   - the later, the more calls
#   - the more calls, the more sby_needed w/ offset because of n_duty (obvious)
#   - most calls in summer
# - weak:
#   - the more n_sick, the less n_duty (obvious)
#   - the more calls, the more n_sick (very weak)

In [ ]:
# plot correlations (except for uninformative columns)
sns.pairplot(df.drop(columns=["n_sby", "n_sick", "n_duty", "dafted"]), height=1.5)

## Model

### Characteristics

- autocorrelated data
- time series data

### Tasks

- feature selection (w/ which technic?) (avoid data leakage of data that is not available when planning)
- choose model (only go for one approach, show why others were not pursued)
- avoid overfitting (how?)

### Objectives

- predict on a daily basis the amount of standby drivers efficiently (maximize rate of called in standby drivers)
- minimize days w/ too little drivers (&rarr; dafted drivers needed) (might result in very little days w/ way to many missing drivers, look out for that)
- (there should be seasonal pattern)

### Restrictions

- plan will be created on the 15th for the following month -> last half of month should not be included in training data

### Further Requirements

- discuss feature importance to increase trust in model
- make predictions as interpretable as possible
- detailed failure analysis to asses situations for which model is not suited
    - plot error in histogram (should be gauss if accumulation somewhere inspect those samples and look for commonalities)
    - 